Name: **your name here**

# Task 2: Toy models

In [ ]:
# Let's start again with providing configurations for convenient output
# Show all output of one executed cell:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In this task you will be constructing your first model.  
We call it toy model for a reason, as you will see the models to be constructed are very simplistic.  
The goal of this task is not simulating real models, but getting to know important COBRA model properties, tools to use for simulation as well as acquiring the knowledge for how to form, simulate, manipulate and again, simulate metabolic models.

---

Remember that whenever you are looking for a function or property that everything is described in the COBRA online description at https://cobrapy.readthedocs.io/en/latest/

## Task 2.0 Loading libraries
In the cell below
- load all packages/classes we need for running COBRA tasks
- show the current cobra configuration
 

In [2]:
## your code ##
## load packages
import cobra
from cobra.io import load_model
## show current cobra configuration
## --- ##

cobra_config = cobra.Configuration()
cobra_config

Attribute,Description,Value
solver,Mathematical optimization solver,glpk
tolerance,"General solver tolerance (feasibility, integrality, etc.)",1e-07
lower_bound,Default reaction lower bound,-1000.0
upper_bound,Default reaction upper bound,1000.0
processes,Number of parallel processes,7
cache_directory,Path for the model cache,/Users/ihe027/Library/Caches/cobrapy
max_cache_size,Maximum cache size in bytes,104857600
cache_expiration,Model cache expiration time in seconds (if any),None


---

### Side note: Nameing convention

Read the note at https://cobrapy.readthedocs.io/en/latest/building_model.html about "Side note: SId"

We use the following naming convention:

- the IDs of any defined model property **cannot(!)** start with a number
- we recommend to use the following *conventions* where you fill in the required information (\<name\>: e.g. brief metabolite or reaction name, \<compartment-abbreviation\>: one letter abbreviation for the compartment, e.g. "e" for extracellular)
  - metabolites: "M_\<name\>_\<compartment-abbreviation\>"
  - reaction: "R_\<name\>_\<compartment-abbreviation\>"

## Task 2.1 Model M1
Consider the following simple model M<sub>1</sub>:

[<img src="toy_models_m1.svg" width="300"/>](toy_models_m1.png "Toymodels")

M<sub>1</sub> is super simple:  
one metabolite A is imported into the model, transformed to B and exported again out of the model.  

Note the compartments *i* and *e*. Compartment "i" is often also termed simply "c" for "cytosol".  
Check the documentation for how to create models at https://cobrapy.readthedocs.io/en/latest/building_model.html.

You have to add 
- 2(4) Metabolites* 
- 3 Reactions
  - consider the correct compartments
  - use meaningful reaction names
  - use standard reaction flux boundaries (typically this is 1000)
    - pay attention to reaction reversibility (reversible reactions have only one arrow head)
  - note that consumed metabolites have negative stoichiometry, while produced metabolites have positive stoichiometry
 
Note: Reactions in COBRA models that *import* or *export* metabolites are called **Exchange Reactions** and define the boundaries of the model. They are typically abbreviated with EX_foo, where foo is the name of a metabolite. There is a special COBRA function for setting up boundary reactions (cobra documentation 3.4.). 

\* Each compartment should have its own metabolite version. Compartment association can be abbreviated with e.g. "_e" suffix for *extracellular*

---

In [3]:
## your code ## 
## create model 
from cobra import Model, Reaction as Rxn, Metabolite as Met
m1 = Model('M1')
m1
## --- ##

Name,M1
Memory address,134a76c90
Number of metabolites,0
Number of reactions,0
Number of genes,0
Number of groups,0
Objective expression,0
Compartments,


In [5]:
## your code ## 
## create metabolites 
## Do not yet add the metabolites to the model.
## --- ##

metA_e = Met(id = "A_e", compartment="e")
metB_e = Met(id = "B_e", compartment="e")
metA_c = Met(id = "A_c", compartment="c")
metB_c = Met(id = "B_c")
metB_c.compartment = "c"


metA_c


Metabolite identifier,A_c
Name,
Memory address,0x134a83790
Formula,None
Compartment,c
In 0 reaction(s),


In [6]:
## your code ##
## add model reactions ##
## Do not yet add the reactions to the model.
## reaction 1

A_in = Rxn(id="R_A_in")
A_in.lower_bound = -1000
A_in.upper_bound = 1000
A_in.add_metabolites({
    metA_e: -1.0,
    metA_c:  1.0
})


## reaction 2
R1 = Rxn(id="R1",          
         lower_bound=-1000,
         upper_bound=1000)
R1.add_metabolites({
    metA_c: -1.0,
    metB_c:  1.0
})

## reaction 3

B_out = Rxn(id="R_B_out")

B_out.lower_bound = -1000
B_out.upper_bound = 1000
B_out.add_metabolites({
    metB_c: -1.0,
    metB_e:  1.0    
})

## --- ##


A_in
R1
B_out


Reaction identifier,R_B_out
Name,
Memory address,0x134a991d0
Stoichiometry,B_c <=> B_e <=>
GPR,
Lower bound,-1000
Upper bound,1000


In [8]:
## your code ##
## Add all reactions to the model.
## Note that metabolites are automatically added if they are correctly added as arguments to the reactions 
## --- ##

m1.add_reactions([A_in, B_out, R1])

m1.add_boundary(metabolite = m1.metabolites.get_by_id("A_e"), type="exchange", ub = 0)
m1.add_boundary(metabolite = m1.metabolites.get_by_id("B_e"), type="exchange", lb = 0)


Ignoring reaction 'R_A_in' since it already exists.
Ignoring reaction 'R_B_out' since it already exists.
Ignoring reaction 'R1' since it already exists.


Reaction identifier,EX_B_e
Name,exchange
Memory address,0x134a9ac10
Stoichiometry,B_e --> -->
GPR,
Lower bound,0
Upper bound,1000.0


In [11]:
## your code ##
## print m1 summary ##
cobra.Model.summary(m1)
## Iterate through all reactions and print their name, lower and upper boundary
print("Reactions")
print("---------")
[print("%s : %s : %s" % (r.id, r.lower_bound, r.upper_bound)) for r in m1.reactions]
## make a copy(!) of m1 called m_baseM1
## --- ##

m_baseM1 = m1.copy()

Non-linear or non-reaction model objective. Falling back to minimal display.


Reactions
---------
R_A_in : -1000 : 1000
R_B_out : -1000 : 1000
R1 : -1000 : 1000
EX_A_e : -1000.0 : 0
EX_B_e : 0 : 1000.0


As we have our model M<sub>1</sub> defined now, we can start simuling.

---

### Task 2.1.1 first simulations
1. Change objective to the outflux of B
2. optimise it using the "optimise function

In [18]:

## your code ##
## change objective
m1.objective = "EX_B_e"
## print again model summary
m1.summary()
## optimize

m1.optimize()
## --- ##


,fluxes,reduced_costs
R_A_in,1000.0,0.0
R_B_out,1000.0,0.0
R1,1000.0,2.0
EX_A_e,-1000.0,0.0
EX_B_e,1000.0,0.0


### Task 2.1.2 change model properties

Now change your model properties and run again:

3. Change lower_bound of R1 to 100
4. optimise   

In [22]:
## your code ##
m1.reactions.R1.upper_bound = 100

m1.reactions.R1.lower_bound = 100
m1.optimize()
## --- ##

,fluxes,reduced_costs
R_A_in,100.0,0.0
R_B_out,100.0,0.0
R1,100.0,2.0
EX_A_e,-100.0,0.0
EX_B_e,100.0,0.0


**NOTE:**  
Notice the "reduced_costs" column of the <code>optimize()</code>? This is an integer linear programming property.  
It simply means that the objective function value would improve by the same factor, if the flux of R1 is allowed to increase by the same factor.
Which value is non-zero?

**OPTIONAL TASK:**
Change the flux through R1 to 2000. Optimise and analyse reduced_costs. Follow the information until you see an increase in the objective function flux to 2000.


In [24]:
## OPTIONAL TASK ##
## your code ##

m1.reactions.R1.upper_bound = 2000
m1.reactions.R_B_out.upper_bound = 2000
m1.reactions.R_A_in.upper_bound = 2000
m1.reactions.EX_A_e.lower_bound = -2000
m1.reactions.EX_B_e.upper_bound = 2000
m1.optimize()
## --- ##

,fluxes,reduced_costs
R_A_in,2000.0,0.0
R_B_out,2000.0,2.0
R1,2000.0,0.0
EX_A_e,-2000.0,0.0
EX_B_e,2000.0,0.0


You have learned basic principles of optimisation until this point.  
By running the <code>optimise()</code>, or <code>slim_optimise()</code> functions, you ran your first flux balance analysis (FBA).  
<code>slim_optimise()</code> is faster than <code>optimise()</code>, but only produces the objective function flux.  
Now, let's extend this model and learn about flux boundary analysis or more commonly called Flux variablity analysis (FVA).

---

---

## Task 2.2. Model M2

1. Create Model M<sub>2</sub> by using standard flux boundaries
2. Simulate it with FBA

*Hint:* Create M<sub>2</sub> as copy of m_baseM1 using the <code>copy()</code> function of models. This prevents you starting from scratch building your exercise model.

*Note:* You have to explicitly define M<sub>2</sub> as a copy of M<sub>1</sub>, otherwise you "only" create a pointer to M<sub>1</sub> instead of a real copy.  
What is the difference to creating a real copy?

[<img src="toy_models_m1-2.svg" width="400"/>](toy_models_m1-2.png "Toymodels M1 & M2")

In [27]:
## your code ##
## Add metabolites and reactions according to the scheme to M2
## create model
m2 = m_baseM1.copy()
m2.id = "M2"
## create and add additional metabolite
metC_c = Met( id = "C_c", compartment="c" )
m2.add_metabolites( metC_c )
m2.reactions.R1.bounds = (0,1000)

## create and add additional reactions

R2=Rxn(id="R2", subsystem="c", lower_bound=0)
R3=Rxn(id="R3", subsystem="c", lower_bound=-1000)
R2.add_metabolites({
    metA_c: -1.0,
    metC_c:  1.0
})

R3.add_metabolites({
    metB_c: 1.0,
    metC_c: -1.0
})

m2.add_reactions([R2, R3])

## add objective to B outflux again

m2.objective = "EX_B_e"

## make a deep copy of this model also termed m_baseM2

m_baseM2 = m2.copy()
## --- ##

In [29]:
## your code ##
## Print again - and control - that all reactions are defined properly
## Pay attention to reaction reversibility ##
## show and check model

print("Reactions")
print("---------")
[print("%s : %s : %s" % (r.id, r.lower_bound, r.upper_bound)) for r in m2.reactions]
## --- ##

m2.summary()

Reactions
---------
R_A_in : -1000 : 1000
R_B_out : -1000 : 1000
R1 : 0 : 1000
EX_A_e : -1000.0 : 0
EX_B_e : 0 : 1000.0
R2 : 0 : 1000.0
R3 : -1000 : 1000.0


Metabolite,Reaction,Flux,C-Number,C-Flux
A_e,EX_A_e,1000,0,0.00%
Metabolite,Reaction,Flux,C-Number,C-Flux
B_e,EX_B_e,-1000,0,0.00%


### Task 2.2.1 Objective function

Set the objective function again to B production and optimize.  
Describe what you observe.

In [30]:
## your code ##
m2.objective = "EX_B_e"
m2.optimize()
## --- ##

,fluxes,reduced_costs
R_A_in,1000.0,0.0
R_B_out,1000.0,2.0
R1,1000.0,0.0
EX_A_e,-1000.0,0.0
EX_B_e,1000.0,0.0
R2,0.0,0.0
R3,0.0,0.0


Do all functions carry flux? If not, why not?  
How do you interpret this?
>
> #### Your answer here
>


---

### Task 2.2.2 flux bounds

Now change the allowed upper bound (outflux or production flux) for R1 to 400 and optimize again.  

In [31]:
## your code ##
m2.reactions.R1.upper_bound = 400
m2.optimize()
## --- ##

,fluxes,reduced_costs
R_A_in,1000.0,0.0
R_B_out,1000.0,2.0
R1,400.0,0.0
EX_A_e,-1000.0,0.0
EX_B_e,1000.0,0.0
R2,600.0,0.0
R3,600.0,0.0


Describe again briefly, what you observe. 

>
> #### Your answer here
>

--- 

### Task 2.2.3 flux bounds II

Now change the flux in R3 from B to C to 300.  
Optimize.  
Then change the flux from C to B in R3 also to 300 and optimize again.

In [32]:
## your code ##
m2.reactions.R3.lower_bound = -300
m2.optimize()
m2.reactions.R3.upper_bound = 300
m2.optimize()
m2.summary()
## --- ##

Metabolite,Reaction,Flux,C-Number,C-Flux
A_e,EX_A_e,700,0,0.00%
Metabolite,Reaction,Flux,C-Number,C-Flux
B_e,EX_B_e,-700,0,0.00%


What do you observe?

>
> #### Your answer here
>


---

As you observed multiple flux distributions that lead to the same objective function value were feasible.  
This is an unfortunate, but common, property of (genome-scale) metabolic models,a s we commonly have more reactions than metabolites defined (especially, if multiple compartments are defined).  

### Task 2.2.4 M2 reset and check

- reset all M2 reaction bounds to their default values
- control by printing again all reactions

In [33]:
## your code ##
m2.reactions.R1.upper_bound = 1000
m2.reactions.R3.lower_bound = -1000
m2.reactions.R3.upper_bound = 1000
[print("%s : %s : %s : %s" % (r.id, r.lower_bound, r.upper_bound, r.subsystem)) for r in m2.reactions]
## --- ##

R_A_in : -1000 : 1000 : 
R_B_out : -1000 : 1000 : 
R1 : 0 : 1000 : 
EX_A_e : -1000.0 : 0 : 
EX_B_e : 0 : 1000.0 : 
R2 : 0 : 1000.0 : c
R3 : -1000 : 1000 : c


[None, None, None, None, None, None, None]

--- 

#### Task 2.2.4.1 M2 FVA

Now, we use a different kind of analysis that will tell us all feasible flux ranges per reaction towards an objective function:  
**F**lux **V**ariability **A**nalysis (FVA)

- Check the documentation on how to import and run FVA
- Print all feasible flux ranges for all reactions 

In [34]:
## your code ##
from cobra.flux_analysis import flux_variability_analysis as fva

fva(m2)
## --- ##

,minimum,maximum
R_A_in,1000.0,1000.0
R_B_out,1000.0,1000.0
R1,0.0,1000.0
EX_A_e,-1000.0,-1000.0
EX_B_e,1000.0,1000.0
R2,0.0,1000.0
R3,0.0,1000.0


What do you observe?  
Is there any reaction that has to carry flux? Which?  
Can you infer relationships between different reactions? Explain briefly your answer.

>
>#### Your answer here
>


---

#### Task 2.2.4.2 FVA | constrain flux

Now, let us prohibit flux in R3 towards metabolite B by setting the respective flux bound to zero.  
Run FVA again.  
Do you observe any change in reaction essentiality?

In [38]:
## your code ##
m2.reactions.R3.upper_bound=0

fva(m2)

## --- ##

,minimum,maximum
R_A_in,1000.0,1000.0
R_B_out,1000.0,1000.0
R1,1000.0,1000.0
EX_A_e,-1000.0,-1000.0
EX_B_e,1000.0,1000.0
R2,0.0,0.0
R3,0.0,0.0


Which reactions became essential?  
Are there any reactions that are not carrying any flux in any solution? If so, why?  
>
> #### Your answer here
>


---

#### Task 2.2.4.3 FVA | relax obj. function condition

Now, run FVA again, but require only 10% of the optimal fraction.  
Again, briefly describe what you observe.

In [39]:
## your code ##
fva(m2, fraction_of_optimum=0.1)
## --- ##

,minimum,maximum
R_A_in,100.0,1000.0
R_B_out,100.0,1000.0
R1,100.0,1000.0
EX_A_e,-1000.0,-100.0
EX_B_e,100.0,1000.0
R2,0.0,0.0
R3,0.0,0.0


#### Your answer here:

---

In [ ]:
from cobra.flux_analysis import pfba

---

## Task 2.3 Model M3

- Start with model M<sub>2</sub> and save it as copy into M<sub>3</sub>
- Modify and add components according to the given scheme
  - Note that you can manipulate reactions in two ways:
    1. Provide a new reaction formula as String to a given reaction. This overrides all so far defined stoichiometries (careful)
    2. Use the add_metabolites() or subtract_metabolites() function of the given reaction
- Control your model if it resembles the scheme.

Note: Do not forget to pay attention on the correct reaction reversibility status.

[<img src="toy_models.svg" width="500"/>](toy_models.png "Toymodels M1-M3")

### Task 2.3.1 M3 simulation

Now, simulate again FBA and FVA.

Also compute a pFBA solution and an FVA solution with <code>pfba_factor=1</code>.  

In [ ]:
## your code ## 
## optimize with regular FBA and pFBA
## optimize FVA with and without pfba_factor
## --- ##

Do you observe differences in the FBA flux solutions?  
Differences in the FVA solutions?  
Briefly explain.

### Task 2.3.2 M3 modifications

Now, change reaction boundaries.

- Consider the following flux ranges
  - v_in  = [-1000,300]
  - v(R2) = [-1000,0] (A <-> 2C)
  - v(R3) = [0,400]
  - v(R4) = [0,1000]
- Run pFBA and FVA again.

In [ ]:
## your code ## 
## --- ##

What do you observe for the given reactions? How many are essential?  
Explain briefly why.

>
>#### Your answer here
>


---

As the last part for this introduction, we introduce one last missing layer of information: Genes.  
Also, we will learn how genes relate to reactions and how we can analyse perturbations of the given system.

### Task 2.3.3 Add gene information to m3

- Add these 7 genes "gR1_1", "gR1_2", "gR2", "gR3", "gR4_1", "gR4_2", "gR4_3" to the model
- The genes encode the reactions as follows:
  - (gR1_1 or gR1_2): R1 
  - (gR2): R2
  - (gR3): R3 
  - (gR4_1 and gR4_2) or gR4_3: R3 

In [ ]:
## your code ##
## add gene to reaction rules to the model
## simulate FBA by calling m3.optimize()
## --- ##

As you can observe there is no change in the optimized flux as we only included more model information and nothing else.  
Now, let us run some model perturbation and see how it affects the model.  

### Task 2.3.4 single gene/rxn KO studies 

1. reset reaction boundaries
2. simulate 
    1. single gene KO
    2. single reaction KO

Check the documentation again, e.g. here https://cobrapy.readthedocs.io/en/latest/deletions.html

In [ ]:
## your code ##
## reset reaction boundaries again to standard values
## run single gene and reaction deletion
## --- ##

Task 2.3.5 dual gene KO

Finally, also run dual gene deletion.

In [ ]:
## your code ##
## --- ##


Explain briefly your observations.

>
>#### Your answer here:
>


---

# Summary

That is it for today.  
I hope you have enjoyed the session.
You may have noticed that particularly for model M<sub>3</sub> it became not as straightforward anymore to understand the computed flux distributions.  
This model did not have even 10 components of any type - imagine mid- or even large-scaled models with 100s of metabolites, genes and reactions. We rely on the simulation results and it is almost impossible to cross-check any given results.  
Hence, it is important to know the properties of metabolic models defined as constraint-based models and how solutions alter upon model perturbations.

You have learned about FBA and FVA, two very common and popular tools to simulate and analyse fluxes.  
Moreover, you have run single and double gene KOs and understand better the relationship to the encoded reactions.

Tomorrow you will work with a moderately sized model and use the tools you got to know today to simulate and analyse the model in depth.  
We will start with a brief summary from today at tomorrow's session.
If any question from today appears later, feel free to bring up the topic tomorrow moring.